# Предобработка текстов

- удаляем строки, где нет текста
- удаляем строки, где нет кириллицы

---
- маскируем номера, электронные почты и обращения; также заменяем эмодзи на маску
- заменяем отступы (в т.ч. "слипшиеся") одинарным пробелом
- выкидываем тексты длиной менее 5 символов
- токенизируем: извлекаем слова (=последовательности из букв казахского и русского языков) в список

---
- удаляем строки, где количество слов на кириллице < 3
- удаляем строки с дублирующимися последоваельностями слов


In [1]:
from preprocessing import cleanupDataSet, save_data

In [2]:
import numpy as np
import random
import os

def seed_everything(seed_value):
    random.seed(seed_value)
    np.random.seed(seed_value)
    # torch.manual_seed(seed_value)
    os.environ['PYTHONHASHSEED'] = str(seed_value)

    # if torch.cuda.is_available():
    #     torch.cuda.manual_seed(seed_value)
    #     torch.cuda.manual_seed_all(seed_value)
    #     torch.backends.cudnn.deterministic = True
    #     torch.backends.cudnn.benchmark = False

SEED = 42
seed_everything(SEED)

In [3]:
data_folder = os.path.join('data', 'raw')

In [4]:
files = list(filter(lambda f: f.endswith('.csv'), os.listdir(data_folder)))
print(len(files))

54


In [26]:
import pandas as pd

# инициализируем датафрейм, в который будем класть предобработанные данные
df = pd.DataFrame(columns=['uuid', 'clean_comment_text', 'source', 'ts'])

In [27]:
init_nrows = 0 # количество строк до очистки для сравнения
final_nrows = 0 # количество строк после очистки

In [7]:
# Telegram
for f in list(filter(lambda x: x.endswith('.json.csv'), files)):
    print(f)

    # извлекаем название канала и формат (обычное видео или youtube short) из названия "сырого" файла
    seg = f.split('_chat')
    channel_name = seg[0] + '_tg'

    # запускаем чистку
    obj = cleanupDataSet(filename=os.path.join(data_folder, f), comment_col='comment_text')

    init_nrows += obj.init_nrows
    final_nrows += obj.final_nrows

    # добавляем столбец с названием источника
    obj.df['source'] = 'telegram__' + obj.df['channel_name'].str.extract('https://t.me/(.+)')

    # удаляем сообщения от ботов
    obj.df.drop(obj.df[obj.df['from_id'].isin(['user5898182404', 'user553147242'])].index, inplace=True)
    obj.df.drop(obj.df[obj.df['comment_text'].str.contains('Top Players', regex=False)].index, inplace=True)

    # сохраняем
    # save_data(df=obj.df[['uuid', 'clean_comment_text', 'source', 'ts']], name=channel_name, save_to_path='data')

    # добавляем данные в общий датафрейм
    df = pd.concat([df, obj.df[['uuid', 'clean_comment_text', 'source', 'ts']]], axis=0)
    print('Final dataframe updated:', len(df), 'rows')

    print('='*80)

egovpress_chat.json.csv
Rows before the cleanup: 10422
Rows after deleting comments with no text: 10383
Rows after deleting comments after deleting comments potentially translated into two languages: 10383
Rows after deleting comments with no Cyrillic characters: 9875
Applying masking, normalizing whitespaces
Rows after deleting comments with less than 3 words with Cyrillic characters: 8768
Rows after deleting duplicate comments: 8738
Cleanup finished! 
Final dataframe updated: 8738 rows
jurttynbalasy_chat.json.csv
Rows before the cleanup: 6583
Rows after deleting comments with no text: 6036
Rows after deleting comments after deleting comments potentially translated into two languages: 6036
Rows after deleting comments with no Cyrillic characters: 5911
Applying masking, normalizing whitespaces
Rows after deleting comments with less than 3 words with Cyrillic characters: 4977
Rows after deleting duplicate comments: 4960
Cleanup finished! 
Final dataframe updated: 13698 rows
qumash_chat.

In [9]:
# Youtube
for f in list(filter(lambda x: not x.endswith('.json.csv'), files)):
    print(f)

    # извлекаем название канала и формат (обычное видео или youtube short) из названия "сырого" файла
    seg = f.split('_')
    channel_name, format = '_'.join(seg[:-1]), seg[-1].split('.')[0]

    # запускаем чистку
    obj = cleanupDataSet(filename=os.path.join(data_folder, f), comment_col='comment_text')

    init_nrows += obj.init_nrows
    final_nrows += obj.final_nrows

    shorts_suff = ('_shorts' if format=='shorts' else '')

    # добавляем столбец с названием источника
    obj.df['source'] = 'youtube' + shorts_suff + '__' + obj.df['channel']

    # сохраняем
    # save_data(df=obj.df[['uuid', 'clean_comment_text', 'source', 'ts']], name=f'{channel_name}_yt{shorts_suff}', save_to_path='data')

    # добавляем данные в общий датафрейм
    df = pd.concat([df, obj.df[['uuid', 'clean_comment_text', 'source', 'ts']]], axis=0)
    print('Final dataframe updated:', len(df), 'rows')
    
    print('='*80)

ainakaz_shorts.csv
Rows before the cleanup: 303
Rows after deleting comments with no text: 303
Rows after deleting comments after deleting comments potentially translated into two languages: 303
Rows after deleting comments with no Cyrillic characters: 232
Applying masking, normalizing whitespaces
Rows after deleting comments with less than 3 words with Cyrillic characters: 169
Rows after deleting duplicate comments: 165
Cleanup finished! 
Final dataframe updated: 536424 rows
ainakaz_vid.csv
Rows before the cleanup: 6360
Rows after deleting comments with no text: 6359
Rows after deleting comments after deleting comments potentially translated into two languages: 6359
Rows after deleting comments with no Cyrillic characters: 5867
Applying masking, normalizing whitespaces
Rows after deleting comments with less than 3 words with Cyrillic characters: 5164
Rows after deleting duplicate comments: 5035
Cleanup finished! 
Final dataframe updated: 541459 rows
ansagaan_vid.csv
Rows before the cl

In [11]:
print('Initial total number of rows in data:', init_nrows)
print('Total number of rows after preprocessing:', final_nrows)
print('Difference:', init_nrows-final_nrows)

Initial total number of rows in data: 1377086
Total number of rows after preprocessing: 946643
Difference: 430443


In [13]:
df['source'].value_counts()

source
telegram__qumash_kz                      496119
youtube__jurttynbalasy                   195215
youtube__kulpynay_korea                   85203
telegram__tilkespekjoq                    26442
youtube__nurmakhan_mukanuly               20855
youtube__zhanerke_seyfullina              14157
telegram__egovpress                        8738
youtube__tilkespekjoq                      8209
youtube__beisekeyev                        8173
youtube_shorts__marat_oralgazin            6178
youtube__marat_oralgazin                   6177
youtube__narikbi                           5976
youtube_shorts__jurttynbalasy              5740
youtube__mnestydno                         5574
youtube__esquirekazakhstan                 5300
youtube__ainakaz                           5035
youtube__nnnlifetv                         5010
telegram__jurttyn_balasy                   4960
youtube__nomadiyar                         3126
youtube__qazstandup                        2810
youtube__standup_central         

In [15]:
# сохраняем датафрейм с указанием источника (не публикуется) в папку data
save_data(df=df, name=f'all_data_src', save_to_path='data')

2026-05-12 19:05:50	all_data_src__cleaned.csv cleaned data: 943840 rows


In [16]:
# сохраняем финальный датафрейм в папку data

save_data(df=df.drop(columns=['source']), name=f'all_data', save_to_path='data')

2026-05-12 19:06:09	all_data__cleaned.csv cleaned data: 943840 rows
